# Lab 1: Phân tích và xử lý tín hiệu âm thanh số

**Họ và tên**: Nguyễn Thị Linh
**MSSV**: 2351260663
**Lớp**: 65TTNT

### Khởi tạo môi trường

In [ ]:
import os
import numpy as np
import scipy.io.wavfile as wav
import scipy.signal as signal
import matplotlib.pyplot as plt
import librosa
import soundfile as sf

plt.rcParams['figure.figsize'] = (10, 4)

audio_dir = "audio"
figures_dir = "figures"
os.makedirs(audio_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)

### Đọc dữ liệu từ file (Phần A: Đọc và kiểm tra dữ liệu)

In [ ]:
# Đọc trực tiếp từ thư mục audio/
speech_input_wav = os.path.join(audio_dir, "speech_input.wav")
music_input_wav = os.path.join(audio_dir, "music_input.wav")

if not os.path.exists(speech_input_wav) or not os.path.exists(music_input_wav):
    raise FileNotFoundError("Vui lòng tự thêm speech_input.wav và music_input.wav vào thư mục audio/")

def analyze_and_plot(wav_path, name_prefix):
    fs, data = wav.read(wav_path)
    channels = data.shape[1] if data.ndim > 1 else 1
    duration = data.shape[0] / fs
    print(f"[{name_prefix}] Fs: {fs} Hz, Channels: {channels}, Duration: {duration:.3f} s")
    
    if channels > 1:
        mono = data.mean(axis=1)
    else:
        mono = data.astype(float)
        
    # Chuẩn hóa
    x = mono / np.max(np.abs(mono))
    time_axis = np.arange(len(x)) / fs
    return fs, x, time_axis

fs_s, x_s, t_s = analyze_and_plot(speech_input_wav, "Speech")
fs_m, x_m, t_m = analyze_and_plot(music_input_wav, "Music")

### Phần B: Phân tích miền thời gian

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t_s, x_s, lw=0.5)
plt.title("Speech Waveform")
plt.ylabel("Amplitude")
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(t_m, x_m, lw=0.5, color='orange')
plt.title("Music Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
plt.show()

# Zoom vào đoạn 0.5 - 1.0s của Speech
idx_s = int(0.5 * fs_s)
idx_e = int(1.0 * fs_s)
seg_s = x_s[idx_s:idx_e]
t_seg_s = t_s[idx_s:idx_e]

plt.figure(figsize=(10, 4))
plt.plot(t_seg_s, seg_s, lw=0.5)
plt.title("Speech Segment (0.5 - 1.0s)")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True)
plt.tight_layout()
plt.show()

### Phần C: Phân tích miền tần số bằng FFT

In [ ]:
seg_m = x_m[int(0.5*fs_m):int(1.0*fs_m)]
w = np.hamming(len(seg_m))
seg_m_w = seg_m * w

N1 = 2048
X1 = 20 * np.log10(np.maximum(np.abs(np.fft.rfft(seg_m_w, n=N1)), 1e-12))
f1 = np.fft.rfftfreq(N1, 1/fs_m)

N2 = 8192
X2 = 20 * np.log10(np.maximum(np.abs(np.fft.rfft(seg_m_w, n=N2)), 1e-12))
f2 = np.fft.rfftfreq(N2, 1/fs_m)

plt.figure(figsize=(12, 5))
plt.plot(f2, X2, lw=0.5, label='NFFT=8192', color='orange')
plt.plot(f1, X1, lw=0.5, label='NFFT=2048', color='blue', alpha=0.7)
plt.title("Music Spectrum (0.5-1.0s) NFFT Comparison")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.xlim(0, 5000)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Phần D: STFT và Spectrogram

In [ ]:
fl1, hop1 = int(0.010 * fs_s), int(0.005 * fs_s)
fl2, hop2 = int(0.050 * fs_s), int(0.025 * fs_s)

f_s1, t_s1, Z1 = signal.spectrogram(x_s, fs=fs_s, window='hamming', nperseg=fl1, noverlap=fl1-hop1, nfft=1024, mode='magnitude')
f_s2, t_s2, Z2 = signal.spectrogram(x_s, fs=fs_s, window='hamming', nperseg=fl2, noverlap=fl2-hop2, nfft=2048, mode='magnitude')

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.pcolormesh(t_s1, f_s1, 20*np.log10(np.maximum(Z1, 1e-12)), shading='gouraud')
plt.title("10ms Frame (Wideband)")
plt.ylim(0, 8000)
plt.subplot(1, 2, 2)
plt.pcolormesh(t_s2, f_s2, 20*np.log10(np.maximum(Z2, 1e-12)), shading='gouraud')
plt.title("50ms Frame (Narrowband)")
plt.ylim(0, 8000)
plt.tight_layout()
plt.show()

### Phần E: Thí nghiệm cửa sổ (Windowing Experiment)

In [ ]:
fr_len = 1024
fr = x_m[10000:10000+fr_len]
w_rect = np.ones(fr_len)
w_hamm = np.hamming(fr_len)
X_r = 20*np.log10(np.maximum(np.abs(np.fft.rfft(fr*w_rect, n=4096)), 1e-12))
X_h = 20*np.log10(np.maximum(np.abs(np.fft.rfft(fr*w_hamm, n=4096)), 1e-12))
f_w = np.fft.rfftfreq(4096, 1/fs_m)

plt.figure(figsize=(10,4))
plt.plot(f_w, X_r, label='Rectangular', lw=0.7)
plt.plot(f_w, X_h, label='Hamming', lw=0.7)
plt.title("Window Comparison")
plt.xlim(0, 4000)
plt.legend()
plt.grid(True)
plt.show()

### Phần F: Thiết kế và Áp dụng Bộ lọc số

In [ ]:
b_lpf = signal.firwin(201, cutoff=2000, fs=fs_s, window='hamming')
b_hpf = signal.firwin(201, cutoff=1000, fs=fs_m, window='hamming', pass_zero=False)

w_lpf, H_lpf = signal.freqz(b_lpf, worN=4096, fs=fs_s)
w_hpf, H_hpf = signal.freqz(b_hpf, worN=4096, fs=fs_m)

plt.figure(figsize=(12,4))
plt.plot(w_lpf, 20*np.log10(np.maximum(np.abs(H_lpf), 1e-12)), label='LPF 2kHz')
plt.plot(w_hpf, 20*np.log10(np.maximum(np.abs(H_hpf), 1e-12)), label='HPF 1kHz')
plt.title("FIR Filter Responses")
plt.legend()
plt.grid(True)
plt.show()

x_s_lpf = signal.lfilter(b_lpf, [1.0], x_s)
x_m_lpf = signal.lfilter(b_lpf, [1.0], x_m)
x_m_hpf = signal.lfilter(b_hpf, [1.0], x_m)

plt.figure(figsize=(10,4))
orig_spec = 20*np.log10(np.maximum(np.abs(np.fft.rfft(x_s[idx_s:idx_e])), 1e-12))
filt_spec = 20*np.log10(np.maximum(np.abs(np.fft.rfft(x_s_lpf[idx_s:idx_e])), 1e-12))
ff = np.fft.rfftfreq(len(orig_spec)*2-1, 1/fs_s)
plt.plot(ff, orig_spec, label='Original', lw=0.5, alpha=0.7)
plt.plot(ff, filt_spec, label='LPF 2kHz', lw=0.5)
plt.title("Spectrum Before and After LPF")
plt.legend()
plt.grid(True)
plt.show()

### Phần G: Lượng tử hóa, Resampling và Mã hóa

In [ ]:
def quantize(x_in, B):
    q = 2**(B-1)-1
    return np.round(np.clip(x_in, -1, 1)*q)/q

bits = [4, 6, 8, 12, 16]
snrs = []
for B in bits:
    xq = quantize(x_m, B)
    e = xq - x_m
    snr = 10*np.log10(np.sum(x_m**2)/np.sum(e**2))
    snrs.append(snr)
    print(f"SNR {B}-bit: {snr:.2f} dB")

plt.figure(figsize=(8,4))
plt.plot(bits, snrs, marker='o')
plt.title("SNR vs Bit Depth")
plt.xlabel("Bits")
plt.ylabel("SNR (dB)")
plt.grid(True)
plt.show()

x_s_8k = librosa.resample(y=x_s, orig_sr=fs_s, target_sr=8000)

plt.figure(figsize=(10,4))
plt.magnitude_spectrum(x_s, Fs=fs_s, scale='dB', color='blue', alpha=0.5, label='Original')
plt.magnitude_spectrum(x_s_8k, Fs=8000, scale='dB', color='red', alpha=0.8, label='Resampled 8kHz')
plt.title("Resampling Spectrum Comparison")
plt.legend()
plt.show()

sizes = [10.09, 0.92]
labels = ['PCM 16-bit 44.1kHz (10.09MB)', 'MP3 128kbps (0.92MB)']
plt.figure(figsize=(6,6))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140, colors=['lightblue', 'lightgreen'])
plt.title("Compression Ratio (~11:1)")
plt.show()